In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install sentence-transformers

<hr>

# Projektbeitrag (Beispiel)

#### Die Lernenden erstellen aus dem Zusammenwirken verschiedener Sprachmodelle einen möglichen Workflow. Dies kann grundsätzlich auch mit einem Sprachmodell geschehen, wenn im Vorfeld entsprechende Anfragen generiert wurden und das Sprachmodell hierfür geeignet ist.

## Ziel: Niederschwellige Darstellung von Retrieval Augmented Generation (RAG)

> Unter Retrieval-Augmented Generation (RAG) versteht man ein Softwaresystem, welches Information Retrieval mit einem Large Language Model kombiniert. Eine Abfrage, welche an das System gestellt wird, kann hierbei auf Informationen aus (externen) Informationsquellen, Datenbanken oder dem World Wide Web zugreifen statt nur auf die Trainingsdaten des Modells. Dies erhöht die Genauigkeit und Robustheit der generierten Inhalte, indem es die Modelle mit aktuellen und spezifischen Informationen versorgt. Typische Anwendungsfälle sind der Zugriff von Chatbots auf interne (Unternehmens-)Daten oder die Bereitstellung von Sachinformationen, die ausschließlich aus verlässlichen Quellen stammen sollen.
> Quelle: https://de.wikipedia.org/wiki/Retrieval-Augmented_Generation

### Voraussetzungen:

#### Die Lernenden müssen die Materialien zu Embeddings und dem Chatbot kennengelernt haben.

##### Hinweis:<br> Der Code ist sehr einfach gehalten, da in FOS 11 keine Programmierkenntnisse vermittelt werden. Jegliche Verzweigung im Quellcode muss <i>manuell</i> durchgeführt werden.

<hr>

In [ ]:
from sentence_transformers import SentenceTransformer # Import der Funktion pipeline des Moduls transformers
from transformers import pipeline # Import der Funktion pipeline des Moduls transformers

In [ ]:
# Festlegung der Datenquellen deren Inhalt nicht in einem Sprachmodell enthalten sind. 
# In der Praxis stammen diese aus externen Quellen, wie z.B. PDF-Dokumenten oder Webseiten
datenquelle= ["Die KI-Schule hat werktags von 8-15 Uhr geöffnet.", "Das Kollegium besteht aus 80 Lehrkräften.","Sie erreichen uns unter folgender Nummer: 0911-123456.","Die Teilnahme an den Kursen ist kostenlos."]

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
quellenEmbeddings = model.encode(datenquelle)
print(quellenEmbeddings)

In [ ]:
userprompt="Bitte teilen Sie mir mit, wie ich Sie kontaktieren kann."

In [ ]:
userpromptEmbedding=model.encode(userprompt) # Die nummerische Darstellung (Embedding) des Userprompts wird erstellt.
#userpromptEmbedding # Kommentar löschen, falls das Embedding angezeigt werden soll.

In [ ]:
# Berechnung der Ähnlichkeiten zwischen Userprompt und Inhalten der Datenquellen

In [ ]:
similarities0 = model.similarity(userpromptEmbedding, quellenEmbeddings[0])
similarities0

In [ ]:
similarities1 = model.similarity(userpromptEmbedding, quellenEmbeddings[1])
similarities1

In [ ]:
similarities2 = model.similarity(userpromptEmbedding, quellenEmbeddings[2])
similarities2

In [ ]:
similarities3 = model.similarity(userpromptEmbedding, quellenEmbeddings[3])
similarities3

In [ ]:
print("Userprompt: " + userprompt)
print("=========")
print("0: " + datenquelle[0]+ "   --- Ähnlichkeit zu " + str(round(float(similarities0)*100,2)) +"%")
print("1: " + datenquelle[1]+ "   --- Ähnlichkeit zu " + str(round(float(similarities1)*100,2)) +"%")
print("2: " + datenquelle[2]+ "   --- Ähnlichkeit zu " + str(round(float(similarities2)*100,2)) +"%")
print("3: " + datenquelle[3]+ "   --- Ähnlichkeit zu " + str(round(float(similarities3)*100,2)) +"%")

In [ ]:
antwort = datenquelle[2] # Wählen Sie manuell die wahrscheinlichste Datenquelle aus. Diese wird in die Antwort des Sprachmodells einbezogen.

pipe = pipeline("text-generation", model="LiquidAI/LFM2-350M", device=0, max_new_tokens=500, temperature=0)
messages = [
    {"role": "system", "content": "Du bist ein nützlicher Chatbot und bekommst folgende Anfrage des Users: #"+userprompt+"#. Die Antwort auf die Frage lautet:#"+antwort+"#. Gib die Antwort auf die Frage in eigenen Worten passend zur Anfrage wieder. Fasse Dich kurz!"}
]
antwortprompt = pipe(messages)
antwortprompt[0]['generated_text'][-1]['content']